# RHI Live Runtime Notebook

Real runtime shell:

```text
prompt
  ↓
slot_builder_lora_v2 emits contract
  ↓
candidate branches generate answers
  ↓
operational critic scores against contract
  ↓
KRRB resolves Ψ collapse or Ω residue
```

Put this notebook in **Downloads**, beside:

```text
slot_builder_lora_v2/
```

It writes:

```text
rhi_live_runtime_outputs/
  rhi_live_runs.jsonl
  rhi_live_runtime_manifest.json
```

No command line. No web. No `/mnt/data` paths inside the notebook.


In [ ]:
# ============================================================
# CONFIG
# ============================================================
from pathlib import Path

ROOT = Path.cwd()

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
SLOT_ADAPTER_DIR = ROOT / "slot_builder_lora_v2"

OUTPUT_DIR = ROOT / "rhi_live_runtime_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_FILES_ONLY = True
USE_4BIT = False

CONTRACT_MAX_NEW_TOKENS = 700
ANSWER_MAX_NEW_TOKENS = 900

N_BRANCHES = 5
DO_SAMPLE_FOR_BRANCHES = True
BRANCH_TEMPERATURE = 0.55
BRANCH_TOP_P = 0.92

SUPPORT_MIN = 3
MARGIN_MIN = 0.08
OMEGA_SCORE_MIN = 0.42

print("ROOT:", ROOT)
print("SLOT_ADAPTER_DIR:", SLOT_ADAPTER_DIR, SLOT_ADAPTER_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# ============================================================
# IMPORTS / OPTIONAL INSTALL
# ============================================================
INSTALL_MISSING = False

if INSTALL_MISSING:
    import sys
    import subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-U",
        "transformers", "peft", "accelerate", "sentencepiece", "pandas"
    ])

import json
import re
import time
import uuid
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))


In [ ]:
# ============================================================
# CONTRACT HELPERS
# ============================================================
REQUIRED_FIELDS = [
    "family_class",
    "domain_carrier",
    "forbidden_neighbor_carrier",
    "boundary_conditions",
    "preserved_function",
    "failure_modes",
    "witness_readout",
    "residue",
]

STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "on", "for", "with",
    "as", "is", "are", "was", "were", "be", "being", "been", "it", "its",
    "this", "that", "these", "those", "by", "from", "into", "at", "while",
    "what", "when", "where", "why", "how", "which", "who", "whom",
    "one", "two", "three", "do", "does", "did", "not", "no", "yes",
    "can", "could", "should", "would", "will", "may", "might",
}

def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")

def extract_first_json_object(text: str) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    text = str(text).strip()

    try:
        obj = json.loads(text)
        return (obj, None) if isinstance(obj, dict) else (None, "json_not_dict")
    except Exception:
        pass

    cleaned = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        obj = json.loads(cleaned)
        return (obj, None) if isinstance(obj, dict) else (None, "fenced_json_not_dict")
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        try:
            obj = json.loads(text[start:end+1])
            return (obj, None) if isinstance(obj, dict) else (None, "scanned_json_not_dict")
        except Exception as e:
            return None, "json_parse_error: " + str(e)

    return None, "no_json_object_found"

def normalize_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        if s.startswith("{") or ":" in s:
            return [s]
        return [p.strip() for p in re.split(r"[|,;]", s) if p.strip()]
    return [str(x).strip()]

def normalize_contract(c0: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    c0 = c0 or {}
    return {
        "family_class": str(c0.get("family_class", "") or "").strip(),
        "domain_carrier": normalize_list(c0.get("domain_carrier", [])),
        "forbidden_neighbor_carrier": normalize_list(c0.get("forbidden_neighbor_carrier", [])),
        "boundary_conditions": normalize_list(c0.get("boundary_conditions", [])),
        "preserved_function": str(c0.get("preserved_function", "") or "").strip(),
        "failure_modes": normalize_list(c0.get("failure_modes", [])),
        "witness_readout": str(c0.get("witness_readout", "") or "").strip(),
        "residue": c0.get("residue", None),
    }

def contract_complete(c: Optional[Dict[str, Any]]) -> bool:
    if not isinstance(c, dict):
        return False
    c = normalize_contract(c)
    for field in REQUIRED_FIELDS:
        if field == "residue":
            continue
        value = c.get(field)
        if isinstance(value, list):
            if len(value) == 0:
                return False
        elif not str(value or "").strip():
            return False
    return True

def wordset(text: Any) -> set:
    if isinstance(text, list):
        text = " ".join(map(str, text))
    text = str(text).lower()
    toks = re.findall(r"[a-zA-Z0-9_]+", text)
    return {t for t in toks if t not in STOPWORDS and len(t) > 1}

def safe_div(a, b):
    return float(a) / float(b) if b else 0.0

print("contract helpers ready")


In [ ]:
# ============================================================
# LOAD MODEL + SLOT ADAPTER
# ============================================================
if not SLOT_ADAPTER_DIR.exists():
    raise FileNotFoundError("Missing slot adapter folder: " + str(SLOT_ADAPTER_DIR))

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    local_files_only=LOCAL_FILES_ONLY,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    "local_files_only": LOCAL_FILES_ONLY,
    "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
}

if USE_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs["device_map"] = "auto"

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)

if not USE_4BIT and torch.cuda.is_available():
    base_model = base_model.to("cuda")

model = PeftModel.from_pretrained(
    base_model,
    SLOT_ADAPTER_DIR,
    local_files_only=LOCAL_FILES_ONLY,
)

model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"

print("loaded base:", MODEL_NAME)
print("loaded slot adapter:", SLOT_ADAPTER_DIR)
print("device:", device)


In [ ]:
# ============================================================
# CHAT + GENERATION HELPERS
# ============================================================
def render_chat(messages: List[Dict[str, str]], add_generation_prompt: bool = False) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

    nl = chr(10)
    out = []
    for m in messages:
        out.append(m.get("role", "user").upper() + ":" + nl + m.get("content", ""))
    if add_generation_prompt:
        out.append("ASSISTANT:" + nl)
    return (nl + nl).join(out)

def generate_text(
    messages: List[Dict[str, str]],
    max_new_tokens: int,
    do_sample: bool = False,
    temperature: float = 0.0,
    top_p: float = 1.0,
    use_slot_adapter: bool = True,
) -> str:
    prompt_text = render_chat(messages, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
    }

    if do_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = top_p

    with torch.no_grad():
        if use_slot_adapter:
            output_ids = model.generate(**inputs, **gen_kwargs)
        else:
            try:
                with model.disable_adapter():
                    output_ids = model.generate(**inputs, **gen_kwargs)
            except Exception:
                output_ids = model.generate(**inputs, **gen_kwargs)

    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("generation helpers ready")


In [ ]:
# ============================================================
# SLOT BUILDER: Q -> C
# ============================================================
SLOT_SYSTEM = "\n".join([
    "You are the Nexus Slot Constructor.",
    "",
    "Your job is to generate the missing-shape contract before answer selection.",
    "",
    "Do not answer the task.",
    "Do not mention answer choices.",
    "Return strict JSON only.",
    "",
    "The contract must contain:",
    "family_class",
    "domain_carrier",
    "forbidden_neighbor_carrier",
    "boundary_conditions",
    "preserved_function",
    "failure_modes",
    "witness_readout",
    "residue",
    "",
    "Use operational fit, not labels.",
])

def build_slot_user_prompt(prompt: str) -> str:
    return "\n".join([
        "Prompt:",
        prompt,
        "",
        "Generate the missing-shape contract.",
        "",
        "Checklist:",
        "1. Need: occupy the inverse cavity.",
        "2. Function: preserve or redirect the required operation.",
        "3. Boundary: respect constraints.",
        "4. Trap: reject noun/surface-label confusion.",
        "5. Collapse: produce one executable witness/readout.",
        "",
        "Return JSON only.",
    ])

def generate_contract(prompt: str) -> Dict[str, Any]:
    messages = [
        {"role": "system", "content": SLOT_SYSTEM},
        {"role": "user", "content": build_slot_user_prompt(prompt)},
    ]

    raw = generate_text(
        messages,
        max_new_tokens=CONTRACT_MAX_NEW_TOKENS,
        do_sample=False,
        use_slot_adapter=True,
    )

    obj, err = extract_first_json_object(raw)
    contract = normalize_contract(obj) if obj else None

    return {
        "raw_contract": raw,
        "contract": contract,
        "parse_error": err,
        "complete": contract_complete(contract),
    }

print("slot builder ready")


In [ ]:
# ============================================================
# CANDIDATE BRANCHES
# ============================================================
BRANCH_SYSTEM = "\n".join([
    "You are an answer generator inside an RHI runtime.",
    "",
    "Use the provided contract as the operational target.",
    "Answer the user's prompt directly.",
    "Do not output JSON unless the user asked for JSON.",
    "Do not mention internal scoring.",
    "Be precise and do not invent facts.",
])

BRANCH_STYLES = [
    {
        "name": "direct",
        "instruction": "Answer directly with the clearest useful response.",
        "sample": False,
        "temperature": 0.0,
    },
    {
        "name": "operational",
        "instruction": "Answer by identifying what the system does, what operation is preserved, and what boundary matters.",
        "sample": False,
        "temperature": 0.0,
    },
    {
        "name": "contract_fit",
        "instruction": "Answer only through the domain carrier and preserved function in the contract.",
        "sample": False,
        "temperature": 0.0,
    },
    {
        "name": "skeptical",
        "instruction": "Answer while actively rejecting forbidden-neighbor confusion and surface-label traps.",
        "sample": True,
        "temperature": BRANCH_TEMPERATURE,
    },
    {
        "name": "residue_aware",
        "instruction": "Answer, then briefly flag any unresolved residue or uncertainty if present.",
        "sample": True,
        "temperature": BRANCH_TEMPERATURE,
    },
]

def branch_user_prompt(prompt: str, contract: Dict[str, Any], style: Dict[str, Any]) -> str:
    return (
        "User prompt:\n"
        + prompt
        + "\n\nMissing-shape contract:\n"
        + json.dumps(normalize_contract(contract), ensure_ascii=False, indent=2)
        + "\n\nBranch instruction:\n"
        + style["instruction"]
        + "\n\nReturn the answer only."
    )

def generate_candidate_branches(prompt: str, contract: Dict[str, Any]) -> List[Dict[str, Any]]:
    branches = []
    styles = BRANCH_STYLES[:N_BRANCHES]

    for style in styles:
        messages = [
            {"role": "system", "content": BRANCH_SYSTEM},
            {"role": "user", "content": branch_user_prompt(prompt, contract, style)},
        ]

        text = generate_text(
            messages,
            max_new_tokens=ANSWER_MAX_NEW_TOKENS,
            do_sample=bool(style["sample"] and DO_SAMPLE_FOR_BRANCHES),
            temperature=float(style["temperature"]),
            top_p=BRANCH_TOP_P,
            use_slot_adapter=False,
        )

        branches.append({
            "branch": style["name"],
            "instruction": style["instruction"],
            "answer": text,
        })

    return branches

print("candidate branch generator ready")


In [ ]:
# ============================================================
# OPERATIONAL CRITIC
# ============================================================
def overlap_score(answer_text: str, terms: Any) -> Dict[str, Any]:
    a = wordset(answer_text)
    t = wordset(terms)
    hits = sorted(a.intersection(t))
    return {
        "score": safe_div(len(hits), len(t)),
        "hits": hits,
        "n_terms": len(t),
        "n_hits": len(hits),
    }

def answer_quality_proxy(answer_text: str) -> float:
    words = re.findall(r"[a-zA-Z0-9_]+", str(answer_text))
    if not words:
        return 0.0
    return min(1.0, len(words) / 120.0)

def score_candidate(prompt: str, contract: Dict[str, Any], candidate: Dict[str, Any]) -> Dict[str, Any]:
    c = normalize_contract(contract)
    answer = candidate["answer"]

    domain = overlap_score(answer, c["domain_carrier"])
    function = overlap_score(answer, c["preserved_function"])
    witness = overlap_score(answer, c["witness_readout"])
    boundary = overlap_score(answer, c["boundary_conditions"])
    forbidden = overlap_score(answer, c["forbidden_neighbor_carrier"])
    prompt_fit = overlap_score(answer, prompt)
    quality = answer_quality_proxy(answer)

    support_flags = {
        "domain": domain["score"] >= 0.20,
        "function": function["score"] >= 0.20,
        "witness": witness["score"] >= 0.20,
        "boundary": boundary["score"] >= 0.15,
        "prompt": prompt_fit["score"] >= 0.15,
        "quality": quality >= 0.35,
    }

    support = int(sum(1 for v in support_flags.values() if v))

    psi = (
        0.30 * domain["score"]
        + 0.22 * function["score"]
        + 0.18 * witness["score"]
        + 0.15 * boundary["score"]
        + 0.15 * prompt_fit["score"]
        + 0.10 * quality
        - 0.25 * forbidden["score"]
    )

    return {
        "branch": candidate["branch"],
        "psi": float(psi),
        "support": support,
        "support_flags": support_flags,
        "scores": {
            "domain": domain,
            "function": function,
            "witness": witness,
            "boundary": boundary,
            "prompt_fit": prompt_fit,
            "forbidden": forbidden,
            "quality": quality,
        },
        "answer": answer,
    }

def score_all_candidates(prompt: str, contract: Dict[str, Any], candidates: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for cand in candidates:
        s = score_candidate(prompt, contract, cand)
        rows.append({
            "branch": s["branch"],
            "psi": s["psi"],
            "support": s["support"],
            "domain": s["scores"]["domain"]["score"],
            "function": s["scores"]["function"]["score"],
            "witness": s["scores"]["witness"]["score"],
            "boundary": s["scores"]["boundary"]["score"],
            "prompt_fit": s["scores"]["prompt_fit"]["score"],
            "forbidden": s["scores"]["forbidden"]["score"],
            "quality": s["scores"]["quality"],
            "answer": s["answer"],
            "detail": s,
        })
    return pd.DataFrame(rows).sort_values(["psi", "support"], ascending=False).reset_index(drop=True)

print("operational critic ready")


In [ ]:
# ============================================================
# KRRB RESOLVER
# ============================================================
def krrb_resolve(score_df: pd.DataFrame, contract_ok: bool) -> Dict[str, Any]:
    if not contract_ok:
        return {
            "state": "Ω",
            "reason": "contract_incomplete_or_unparseable",
            "winner": None,
            "margin": None,
            "support": 0,
        }

    if score_df.empty:
        return {
            "state": "Ω",
            "reason": "no_candidates",
            "winner": None,
            "margin": None,
            "support": 0,
        }

    top = score_df.iloc[0].to_dict()
    second_psi = float(score_df.iloc[1]["psi"]) if len(score_df) > 1 else 0.0
    margin = float(top["psi"] - second_psi)
    support = int(top["support"])

    fail_reasons = []
    if support < SUPPORT_MIN:
        fail_reasons.append("support_below_min")
    if margin < MARGIN_MIN:
        fail_reasons.append("margin_below_min")
    if float(top["psi"]) < OMEGA_SCORE_MIN:
        fail_reasons.append("psi_below_min")

    if fail_reasons:
        return {
            "state": "Ω",
            "reason": " | ".join(fail_reasons),
            "winner": top,
            "margin": margin,
            "support": support,
        }

    return {
        "state": "Ψ",
        "reason": "collapse",
        "winner": top,
        "margin": margin,
        "support": support,
    }

def build_omega_report(prompt: str, contract_result: Dict[str, Any], score_df: pd.DataFrame, resolution: Dict[str, Any]) -> Dict[str, Any]:
    contract = contract_result.get("contract")
    top = resolution.get("winner")

    return {
        "omega_id": "omega_" + uuid.uuid4().hex[:10],
        "time": now_iso(),
        "prompt": prompt,
        "reason": resolution.get("reason"),
        "contract_parse_error": contract_result.get("parse_error"),
        "contract_complete": contract_result.get("complete"),
        "contract": contract,
        "top_branch": None if top is None else top.get("branch"),
        "top_psi": None if top is None else top.get("psi"),
        "top_support": resolution.get("support"),
        "margin": resolution.get("margin"),
        "candidate_scores": score_df.drop(columns=["detail"]).to_dict(orient="records") if not score_df.empty else [],
    }

print("KRRB resolver ready")


In [ ]:
# ============================================================
# LIVE PROMPT — edit this
# ============================================================
LIVE_PROMPT = """
Using the Nexus lens, explain why current AI agents fail when they use tools before forming a contract.
"""

print(LIVE_PROMPT.strip())


In [ ]:
# ============================================================
# RUN RHI ON ONE PROMPT
# ============================================================
def run_rhi(prompt: str, save: bool = True) -> Dict[str, Any]:
    run_id = "rhi_" + uuid.uuid4().hex[:10]
    t0 = time.time()

    print("Δ generating contract...")
    contract_result = generate_contract(prompt)
    contract = contract_result["contract"]

    print("contract complete:", contract_result["complete"])
    if contract is not None:
        print(json.dumps(contract, ensure_ascii=False, indent=2)[:2500])
    else:
        print("raw contract:")
        print(contract_result["raw_contract"])

    if not contract_result["complete"]:
        empty_df = pd.DataFrame()
        resolution = krrb_resolve(empty_df, contract_ok=False)
        omega = build_omega_report(prompt, contract_result, empty_df, resolution)
        result = {
            "run_id": run_id,
            "time": now_iso(),
            "prompt": prompt,
            "contract_result": contract_result,
            "candidates": [],
            "scores": [],
            "resolution": resolution,
            "answer": None,
            "omega": omega,
            "elapsed_sec": time.time() - t0,
        }
    else:
        print("Δ generating candidate branches...")
        candidates = generate_candidate_branches(prompt, contract)

        print("Δ scoring candidates...")
        score_df = score_all_candidates(prompt, contract, candidates)
        display(score_df.drop(columns=["detail"]))

        resolution = krrb_resolve(score_df, contract_ok=True)

        if resolution["state"] == "Ψ":
            answer = resolution["winner"]["answer"]
            omega = None
            print("Ψ collapse:", resolution["winner"]["branch"], "margin:", round(resolution["margin"], 4), "support:", resolution["support"])
            print()
            print(answer)
        else:
            answer = None
            omega = build_omega_report(prompt, contract_result, score_df, resolution)
            print("Ω residue:", resolution["reason"])
            print(json.dumps(omega, ensure_ascii=False, indent=2)[:2500])

        result = {
            "run_id": run_id,
            "time": now_iso(),
            "prompt": prompt,
            "contract_result": contract_result,
            "candidates": candidates,
            "scores": score_df.drop(columns=["detail"]).to_dict(orient="records"),
            "resolution": resolution,
            "answer": answer,
            "omega": omega,
            "elapsed_sec": time.time() - t0,
        }

    if save:
        out_path = OUTPUT_DIR / "rhi_live_runs.jsonl"
        with out_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(result, ensure_ascii=False, default=str) + chr(10))
        print()
        print("saved:", out_path)

    return result

live_result = run_rhi(LIVE_PROMPT.strip(), save=True)


In [ ]:
# ============================================================
# BATCH MODE — optional
# ============================================================
PROMPTS = [
    "Why does RAG fail when retrieval happens before intent is stabilized?",
    "Explain LoRA as a groove in a frozen model manifold using Nexus terms.",
    "What does residue repair add to a normal AI agent loop?",
]

RUN_BATCH = False

if RUN_BATCH:
    batch_results = []
    for i, prompt in enumerate(PROMPTS):
        print("=" * 100)
        print("BATCH", i + 1, "/", len(PROMPTS))
        batch_results.append(run_rhi(prompt, save=True))
    print("batch complete:", len(batch_results))
else:
    print("batch skipped")


In [ ]:
# ============================================================
# READ SAVED RUNS / RESIDUE MEMORY
# ============================================================
def read_saved_runs(path: Path) -> List[Dict[str, Any]]:
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

runs_file = OUTPUT_DIR / "rhi_live_runs.jsonl"
saved_runs = read_saved_runs(runs_file)

summary_rows = []
for r in saved_runs:
    res = r.get("resolution", {})
    summary_rows.append({
        "run_id": r.get("run_id"),
        "time": r.get("time"),
        "state": res.get("state"),
        "reason": res.get("reason"),
        "support": res.get("support"),
        "margin": res.get("margin"),
        "elapsed_sec": r.get("elapsed_sec"),
        "prompt": str(r.get("prompt", ""))[:180],
    })

runs_df = pd.DataFrame(summary_rows)
display(runs_df)

manifest = {
    "notebook": "rhi_live_runtime_notebook",
    "model_name": MODEL_NAME,
    "slot_adapter_dir": str(SLOT_ADAPTER_DIR),
    "output_dir": str(OUTPUT_DIR),
    "runs_file": str(runs_file),
    "n_saved_runs": len(saved_runs),
    "runtime_shape": "Q -> C -> {A_i} -> Ψ/Ω",
    "collapse_controls": {
        "support_min": SUPPORT_MIN,
        "margin_min": MARGIN_MIN,
        "omega_score_min": OMEGA_SCORE_MIN,
    }
}

(OUTPUT_DIR / "rhi_live_runtime_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print(json.dumps(manifest, indent=2))


# Ψ-collapse

This is the first real RHI runtime shell:

```text
slot_builder_lora_v2 = contract organ
base Qwen = answer organ
critic = operational fit gate
KRRB = collapse/residue resolver
JSONL log = residue memory
```

Current limitation:

```text
critic is lexical/structural
```

Next real advance:

```text
mine rhi_live_runs.jsonl
  ↓
convert Ω records into repair rows
  ↓
train operational critic / residue repairer
```

The slot-builder is no longer the project. It is now one organ inside the runtime.
